In [1]:
from pathlib import Path
import pandas as pd

# Define data path / Pfad definieren
DATA_DIR = Path("./data")

train_df = pd.read_csv(
    DATA_DIR / "train.csv", parse_dates=["date"], nrows=500_000
)

items_df = pd.read_csv(DATA_DIR / "items.csv")
stores_df = pd.read_csv(DATA_DIR / "stores.csv")
oil_df = pd.read_csv(DATA_DIR / "oil.csv", parse_dates=["date"])
holidays_df = pd.read_csv(DATA_DIR / "holidays_events.csv", parse_dates=["date"])


# 1. Fill missing oil prices using the previous day's price (Forward Fill)
# Fehlende Ölpreise mit dem Wert des Vortages auffüllen (Forward Fill)
oil_df["dcoilwtico"] = oil_df["dcoilwtico"].ffill().bfill()

# 2. Filter out transferred holidays to keep only active ones
# Verschobene Feiertage herausfiltern, um nur aktive zu behalten
active_holidays = holidays_df[holidays_df["transferred"] == False]
holiday_flags = (
    active_holidays.groupby("date")
    .agg(is_holiday=("type", "count"))
    .reset_index()
)

# English: Merge everything into the main training set
# Deutsch: Alles in den Haupt-Trainingsdatensatz zusammenführen
# ==========================================
train_merged = train_df.merge(items_df, on="item_nbr", how="left")
train_merged = train_merged.merge(stores_df, on="store_nbr", how="left")
train_merged = train_merged.merge(oil_df, on="date", how="left")
train_merged = train_merged.merge(holiday_flags, on="date", how="left")

train_merged["is_holiday"] = train_merged["is_holiday"].fillna(0).astype(int)

print("Merged Data Shape:", train_merged.shape)
train_merged.head()

Merged Data Shape: (500000, 15)


,id,date,store_nbr,item_nbr,unit_sales,onpromotion,family,class,perishable,city,state,type,cluster,dcoilwtico,is_holiday
0,0,2013-01-01,25,103665,7.0,NaN,BREAD/BAKERY,2712,1,Salinas,Santa Elena,D,1,93.14,1
1,1,2013-01-01,25,105574,1.0,NaN,GROCERY I,1045,0,Salinas,Santa Elena,D,1,93.14,1
2,2,2013-01-01,25,105575,2.0,NaN,GROCERY I,1045,0,Salinas,Santa Elena,D,1,93.14,1
3,3,2013-01-01,25,108079,1.0,NaN,GROCERY I,1030,0,Salinas,Santa Elena,D,1,93.14,1
4,4,2013-01-01,25,108701,1.0,NaN,DELI,2644,1,Salinas,Santa Elena,D,1,93.14,1


In [18]:
train_merged["year"] = train_merged["date"].dt.year
train_merged["month"] = train_merged["date"].dt.month
train_merged["day"] = train_merged["date"].dt.day
train_merged["dayofweek"] = train_merged["date"].dt.dayofweek

In [19]:
categorical_cols = ["family", "perishable", "city", "state", "type", "cluster"]
for col in categorical_cols:
  if col in train_merged.columns:
    train_merged[col] = train_merged[col].astype("category")

print("Feature engineering complete!")

Feature engineering complete!


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

drop_cols = ["id", "date", "unit_sales"]
feature_cols = [col for col in train_merged.columns if col not in drop_cols]


X = train_merged[feature_cols].copy()
X = pd.get_dummies(X, drop_first=True)

y = train_merged["unit_sales"].fillna(0)


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


print("Training the model... Please wait.")
model = RandomForestRegressor(
    n_estimators=10,  # Small number of trees for faster beginner training / Geringe Anzahl an Bäumen für schnelleres Training
    max_depth=10,  # Limit depth to prevent overfitting / Begrenzte Tiefe zur Vermeidung von Overfitting
    random_state=42,
    n_jobs=-1,  # Use all CPU cores / Alle CPU-Kerne nutzen
)

model.fit(X_train, y_train)
print("Model training completed successfully!")

val_predictions = model.predict(X_val)

# Ensure no negative predictions (as required by the competition rule)
# Sicherstellen, dass keine negativen Vorhersagen getroffen werden (gemäß Wettbewerbsregeln)
val_predictions = val_predictions.clip(min=0)

print("First 5 predictions on validation set:", val_predictions[:5].round(2))

Training the model... Please wait.
Model training completed successfully!
First 5 predictions on validation set: [10.32 15.44  5.97  7.38  7.84]


In [23]:
# Load test set / Testset laden
test_df = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])

# Apply the same merge pipeline used for training
# Dasselbe Merge-Pipeline wie beim Training anwenden
test_merged = test_df.merge(items_df, on="item_nbr", how="left")
test_merged = test_merged.merge(stores_df, on="store_nbr", how="left")
test_merged = test_merged.merge(oil_df, on="date", how="left")
test_merged = test_merged.merge(holiday_flags, on="date", how="left")

# Fill missing holiday flags with 0 / Fehlende Feiertags-Flags mit 0 füllen
test_merged["is_holiday"] = test_merged["is_holiday"].fillna(0).astype(int)

# Extract date features / Datumsmerkmale extrahieren
test_merged["year"] = test_merged["date"].dt.year
test_merged["month"] = test_merged["date"].dt.month
test_merged["day"] = test_merged["date"].dt.day
test_merged["dayofweek"] = test_merged["date"].dt.dayofweek

# Convert categorical columns / Kategorische Spalten umwandeln
for col in categorical_cols:
  if col in test_merged.columns:
    test_merged[col] = test_merged[col].astype("category")

# ==========================================
# Align Test Features with Training Features
# Testmerkmale mit Trainingsmerkmalen abgleichens
# ==========================================
X_test = test_merged[feature_cols].copy()
X_test = pd.get_dummies(X_test, drop_first=True)

# Ensure test set has the exact same columns as the training set
# Sicherstellen, dass das Testset genau dieselben Spalten wie das Trainingsset hat
X_test = X_test.reindex(columns=X.columns, fill_value=0)

# ==========================================
# Make Predictions and Format Submission File
# Vorhersagen treffen und Einreichungsdatei formatieren
# ==========================================
print("Generating predictions for the test set...")
test_predictions = model.predict(X_test)

# Competition rule: Ensure no negative predictions because of log transformation
# Wettbewerbsregel: Sicherstellen, dass keine negativen Vorhersagen wegen Log-Transformation vorliegen
test_predictions = test_predictions.clip(min=0)

# Create submission dataframe matching the required format (id, unit_sales)
# Erstellen des Submission-Dataframes im geforderten Format (id, unit_sales)[cite: 1]
submission = pd.DataFrame(
    {"id": test_merged["id"], "unit_sales": test_predictions}
)

# Save to CSV file / Als CSV-Datei speichern
submission.to_csv("./data/sample_submission.csv", index=False)
print("Submission file 'submission.csv' created successfully!")
submission.head()

Generating predictions for the test set...
Submission file 'submission.csv' created successfully!


,id,unit_sales
0,125497040,6.085918
1,125497041,5.958598
2,125497042,5.182065
3,125497043,5.607316
4,125497044,5.906839
